In [109]:
import pandas as pd
from sqlalchemy import text
from sqlalchemy import create_engine,text
from urllib.parse import quote_plus
import time
import numpy as np
from datetime import date

In [2]:
connection_string = (
    "DRIVER={ODBC Driver 18 for SQL Server};"
    "SERVER=localhost,1433;"
    "DATABASE=DataWarehouse;"
    "Trusted_Connection=yes;"
    "Encrypt=yes;"
    "TrustServerCertificate=yes;"
)

engine = create_engine(
    "mssql+pyodbc:///?odbc_connect="
    + quote_plus(connection_string)
)

In [23]:
query = text("""
    SELECT TABLE_NAME
    FROM INFORMATION_SCHEMA.TABLES
    WHERE TABLE_SCHEMA = 'raw'
      AND TABLE_TYPE = 'BASE TABLE'
    ORDER BY TABLE_NAME
""")

with engine.connect() as conn:
    tables_df = pd.read_sql(query, conn)

tables_df

,TABLE_NAME
0,crm_cust_info
1,crm_prd_info
2,crm_sales_details
3,erp_cust_az12
4,erp_loc_a101
5,erp_px_cat_g1v2


## CRM Tables

### crm_cust_info

In [24]:
crm_cust_info=pd.read_sql('''SELECT *
                          FROM raw.crm_cust_info''',engine)
crm_cust_info.head()

,cst_id,cst_key,cst_firstname,cst_lastname,cst_marital_status,cst_gndr,cst_create_date
0,11000.0,AW00011000,Jon,Yang,M,M,2025-10-06
1,11001.0,AW00011001,Eugene,Huang,S,M,2025-10-06
2,11002.0,AW00011002,Ruben,Torres,M,M,2025-10-06
3,11003.0,AW00011003,Christy,Zhu,S,F,2025-10-06
4,11004.0,AW00011004,Elizabeth,Johnson,S,F,2025-10-06


In [25]:
## check for duplicates or null id
crm_cust_info['cst_id'].value_counts(dropna=False)

cst_id
NaN        4
29466.0    3
29483.0    2
29449.0    2
29433.0    2
          ..
11048.0    1
11049.0    1
11050.0    1
11051.0    1
11028.0    1
Name: count, Length: 18485, dtype: int64

In [26]:
## data consistency
crm_cust_info['cst_gndr'].value_counts(dropna=False)

cst_gndr
M       7068
F       6848
None    4578
Name: count, dtype: int64

In [27]:
## data consistency
crm_cust_info['cst_marital_status'].value_counts(dropna=False)

cst_marital_status
M       10013
S        8474
None        7
Name: count, dtype: int64

In [ ]:
## take most recent row for each customer
crm_cust_info=crm_cust_info.sort_values(by='cst_create_date', ascending=False).drop_duplicates('cst_id')
## eliminate row with null id
crm_cust_info=crm_cust_info[crm_cust_info['cst_id'].isna()==False]

## clean and normalize values
crm_cust_info['cst_firstname']=crm_cust_info['cst_firstname'].str.strip()
crm_cust_info['cst_lastname']=crm_cust_info['cst_lastname'].str.strip()

crm_cust_info["cst_marital_status"] = (crm_cust_info["cst_marital_status"].fillna("").astype(str).str.strip().str.upper().map({
        "S": "Single",
        "M": "Married",
        "": "Unknown"
    })
    .fillna("error"))

crm_cust_info["cst_gndr"] = (crm_cust_info["cst_gndr"].fillna("").astype(str).str.strip().str.upper().map({
        "F": "Female",
        "M": "Male",
        "": "Unknown"
    })
    .fillna("error")
)

In [30]:
crm_cust_info['cst_marital_status'].value_counts(dropna=False)

cst_marital_status
Married    10011
Single      8473
Name: count, dtype: int64

In [31]:
crm_cust_info['cst_gndr'].value_counts(dropna=False)

cst_gndr
Male       7067
Female     6848
Unknown    4569
Name: count, dtype: int64

In [32]:
crm_cust_info['cst_id'].value_counts(dropna=False)

cst_id
11000.0    1
29483.0    1
29466.0    1
29433.0    1
29473.0    1
          ..
28252.0    1
28253.0    1
28254.0    1
28255.0    1
28256.0    1
Name: count, Length: 18484, dtype: int64

In [33]:
with engine.begin() as conn:
    print('Truncating Table: crm_cust_info')
    conn.execute(text("TRUNCATE TABLE raw.crm_cust_info"))

    print('Loading Table: crm_cust_info')
    crm_cust_info.to_sql(name='crm_cust_info',schema='clean',con=conn,if_exists='append',index=False)


Truncating Table: crm_cust_info
Loading Table: crm_cust_info


### crm_prd_info

In [71]:
crm_prd_info=pd.read_sql('''SELECT *
                          FROM raw.crm_prd_info''',engine)
crm_prd_info.head()

,prd_id,prd_key,prd_nm,prd_cost,prd_line,prd_start_dt,prd_end_dt
0,210,CO-RF-FR-R92B-58,HL Road Frame - Black- 58,NaN,R,2003-07-01,NaT
1,211,CO-RF-FR-R92R-58,HL Road Frame - Red- 58,NaN,R,2003-07-01,NaT
2,212,AC-HE-HL-U509-R,Sport-100 Helmet- Red,12.0,S,2011-07-01,2007-12-28
3,213,AC-HE-HL-U509-R,Sport-100 Helmet- Red,14.0,S,2012-07-01,2008-12-27
4,214,AC-HE-HL-U509-R,Sport-100 Helmet- Red,13.0,S,2013-07-01,NaT


In [72]:
## check for duplicates 
crm_prd_info['prd_id'].value_counts(dropna=False)

prd_id
606    1
210    1
211    1
212    1
213    1
      ..
226    1
227    1
228    1
229    1
230    1
Name: count, Length: 397, dtype: int64

In [73]:
## check for null id
crm_prd_info[crm_prd_info['prd_id'].isna()]

,prd_id,prd_key,prd_nm,prd_cost,prd_line,prd_start_dt,prd_end_dt


In [68]:
## check for negative prices
crm_prd_info[(crm_prd_info['prd_cost'].isna())|
             (crm_prd_info['prd_cost']<0)]

,prd_id,prd_key,prd_nm,prd_cost,prd_line,prd_start_dt,prd_end_dt
0,210,CO-RF-FR-R92B-58,HL Road Frame - Black- 58,NaN,R,2003-07-01,NaT
1,211,CO-RF-FR-R92R-58,HL Road Frame - Red- 58,NaN,R,2003-07-01,NaT


In [69]:
crm_prd_info['prd_line'].value_counts(dropna=False)

prd_line
R       162
M       112
S        54
T        52
None     17
Name: count, dtype: int64

In [70]:
crm_prd_info[(crm_prd_info['prd_id']>409)&(crm_prd_info['prd_id']<417)]

,prd_id,prd_key,prd_nm,prd_cost,prd_line,prd_start_dt,prd_end_dt
200,410,CO-WH-FW-M423,LL Mountain Front Wheel,27.0,M,2012-07-01,2008-12-27
201,411,CO-WH-FW-M762,ML Mountain Front Wheel,93.0,M,2012-07-01,2008-12-27
202,412,CO-WH-FW-M928,HL Mountain Front Wheel,133.0,M,2012-07-01,2008-12-27
203,413,CO-WH-FW-R623,LL Road Front Wheel,38.0,R,2012-07-01,2008-12-27
204,414,CO-WH-FW-R762,ML Road Front Wheel,110.0,R,2012-07-01,2008-12-27
205,415,CO-WH-FW-R820,HL Road Front Wheel,147.0,R,2012-07-01,2008-12-27
206,416,CO-WH-FW-T905,Touring Front Wheel,97.0,T,2012-07-01,2008-12-27


In [75]:
## check invalid dates
crm_prd_info[crm_prd_info['prd_start_dt']>crm_prd_info['prd_end_dt']].sort_values(["prd_key", "prd_start_dt"])

,prd_id,prd_key,prd_nm,prd_cost,prd_line,prd_start_dt,prd_end_dt


In [49]:
crm_prd_info['cat_id']=crm_prd_info['prd_key'].astype(str).str[:5].str.replace('-','_')
crm_prd_info['prd_key']=crm_prd_info['prd_key'].astype(str).str[7:]
crm_prd_info['prd_cost']=crm_prd_info['prd_cost'].fillna(0)
crm_prd_info["prd_line"] = (crm_prd_info["prd_line"].fillna("").astype(str).str.strip().str.upper().map({
        "R": "Road",
        "M": "Mountain",
        "S": "Other Sales",
        "T":"Touring"
    })
    .fillna("error"))

In [77]:
## derivates columns to use as key for other tables
crm_prd_info["cat_id"] = crm_prd_info["prd_key"].astype("string").str[:5].str.replace("-", "_", regex=False)
crm_prd_info["prd_key"] = crm_prd_info["prd_key"].astype("string").str[6:]
## clean and normalize
crm_prd_info["prd_cost"] = crm_prd_info["prd_cost"].fillna(0)
crm_prd_info["prd_line"] = (crm_prd_info["prd_line"].astype("string").str.strip().str.upper()
    .map({
        "M": "Mountain",
        "R": "Road",
        "S": "Other Sales",
        "T": "Touring"
    })
    .fillna("Unknown")
)

In [74]:
##calculate end date as 1 day before next start date
crm_prd_info["prd_start_dt"] = pd.to_datetime(crm_prd_info["prd_start_dt"]).dt.date
crm_prd_info["prd_end_dt"] = pd.to_datetime(crm_prd_info["prd_end_dt"]).dt.date
crm_prd_info=crm_prd_info.sort_values(["prd_key", "prd_start_dt"])
crm_prd_info["prd_end_dt"] = (crm_prd_info.groupby("prd_key")["prd_start_dt"]
                              .shift(-1)
                              .sub(pd.Timedelta(days=1))
                              
)

In [78]:
with engine.begin() as conn:
    print('Truncating Table: crm_prd_info')
    conn.execute(text("TRUNCATE TABLE raw.crm_prd_info"))

    print('Loading Table: crm_prd_info')
    crm_prd_info.to_sql(name='crm_prd_info',schema='clean',con=conn,if_exists='append',index=False)

Truncating Table: crm_prd_info
Loading Table: crm_prd_info


### crm_sales_details

In [97]:
crm_sales_details=pd.read_sql('''SELECT *
                          FROM raw.crm_sales_details''',engine)
crm_sales_details.head()

,sls_ord_num,sls_prd_key,sls_cust_id,sls_order_dt,sls_ship_dt,sls_due_dt,sls_sales,sls_quantity,sls_price
0,SO47081,BK-R50B-48,15203,20120130,20120206,20120211,783.0,1,783.0
1,SO47082,BK-R50R-60,15254,20120130,20120206,20120211,783.0,1,783.0
2,SO47083,BK-R50R-48,15288,20120130,20120206,20120211,783.0,1,783.0
3,SO47084,BK-R89R-52,21561,20120130,20120206,20120211,2443.0,1,2443.0
4,SO47085,BK-R89B-58,16311,20120131,20120207,20120212,2182.0,1,2182.0


In [98]:
## check for invalid dates
crm_sales_details[(crm_sales_details['sls_order_dt'].astype(str).str.len()>8)|
                (crm_sales_details['sls_order_dt']<=0)|
                (crm_sales_details['sls_order_dt']> 20500101)|
                (crm_sales_details['sls_order_dt']<19900101)
                ][['sls_order_dt','sls_ship_dt','sls_due_dt']].drop_duplicates()

,sls_order_dt,sls_ship_dt,sls_due_dt
32847,0,20130823,20130828
32948,0,20130824,20130829
33574,0,20130827,20130901
33883,0,20130829,20130903
34890,0,20130905,20130910
44225,32154,20131102,20131107
44226,5489,20131102,20131107


In [99]:
## check for invalid dates
crm_sales_details[(crm_sales_details['sls_order_dt']>crm_sales_details['sls_ship_dt'])|
                (crm_sales_details['sls_order_dt']>crm_sales_details['sls_due_dt'])
                ][['sls_order_dt','sls_ship_dt','sls_due_dt']].drop_duplicates()

,sls_order_dt,sls_ship_dt,sls_due_dt


In [100]:
###CHECK DATA CONSISTENCY
## sales= quantity * price
## values must not be null, 0 or negative
##expectaion: no reults
crm_sales_details[(crm_sales_details['sls_sales']!=crm_sales_details['sls_price']*crm_sales_details['sls_quantity'])|
                  (crm_sales_details['sls_sales']<=0)|(crm_sales_details['sls_price']<=0)|(crm_sales_details['sls_quantity']<=0)|
                  (crm_sales_details['sls_sales'].isna())|(crm_sales_details['sls_price'].isna())|(crm_sales_details['sls_quantity'].isna())]

,sls_ord_num,sls_prd_key,sls_cust_id,sls_order_dt,sls_ship_dt,sls_due_dt,sls_sales,sls_quantity,sls_price
3186,SO51259,WB-H098,11433,20130101,20130108,20130113,10.0,2,NaN
3298,SO51298,WB-H098,27949,20130104,20130111,20130116,25.0,5,NaN
3546,SO51387,HL-U509-B,11942,20130109,20130116,20130121,70.0,2,NaN
3799,SO51479,BC-R205,16687,20130116,20130123,20130128,9.0,1,NaN
3801,SO51479,HL-U509,16687,20130116,20130123,20130128,35.0,1,NaN
4525,SO51942,BC-M005,11223,20130129,20130205,20130210,100.0,10,NaN
5095,SO52187,CL-9009,18110,20130203,20130210,20130215,16.0,2,NaN
17670,SO57804,BK-M38S-40,16470,20130511,20130518,20130523,769.0,1,-769.0
17951,SO57916,TI-M602,23024,20130513,20130520,20130525,30.0,1,-30.0
18975,SO58326,FE-6654,12045,20130520,20130527,20130601,22.0,1,-22.0


In [101]:
## converts dates
sls_order_dt_str = (crm_sales_details["sls_order_dt"].astype("string").str.strip())
crm_sales_details["sls_order_dt"] = pd.to_datetime(
    sls_order_dt_str.where(
        (sls_order_dt_str != "0") &
        (sls_order_dt_str.str.len() == 8)
    ),format="%Y%m%d",errors="coerce").dt.date

sls_ship_dt_str = (crm_sales_details["sls_ship_dt"].astype("string").str.strip())
crm_sales_details["sls_ship_dt"] = pd.to_datetime(
    sls_ship_dt_str.where(
        (sls_ship_dt_str != "0") &
        (sls_ship_dt_str.str.len() == 8)
    ),format="%Y%m%d",errors="coerce").dt.date

sls_due_dt_str = (crm_sales_details["sls_due_dt"].astype("string").str.strip())
crm_sales_details["sls_due_dt"] = pd.to_datetime(
    sls_due_dt_str.where(
        (sls_due_dt_str != "0") &
        (sls_due_dt_str.str.len() == 8)
    ),format="%Y%m%d",errors="coerce").dt.date

In [102]:
crm_sales_details["sls_sales"] = crm_sales_details.apply(lambda x: 
        x["sls_quantity"] * abs(x["sls_price"]) if pd.isna(x["sls_sales"]) 
            or x["sls_sales"] <= 0
            or x["sls_sales"] != x["sls_quantity"] * abs(x["sls_price"])
        else x["sls_sales"]   ,axis=1
)


In [103]:
crm_sales_details["sls_price"] = crm_sales_details.apply(lambda x: 
        x["sls_sales"] / x["sls_quantity"] if (pd.isna(x["sls_price"]) or x["sls_price"] <= 0)
            and x["sls_quantity"] != 0 else 
        np.nan if (pd.isna(x["sls_price"]) or x["sls_price"] <= 0) else 
        x["sls_price"]     ,axis=1
)

In [104]:
with engine.begin() as conn:
    print('Truncating Table: crm_sales_details')
    conn.execute(text("TRUNCATE TABLE raw.crm_sales_details"))

    print('Loading Table: crm_sales_details')
    crm_sales_details.to_sql(name='crm_sales_details',schema='clean',con=conn,if_exists='append',index=False)

Truncating Table: crm_sales_details
Loading Table: crm_sales_details


## ERP Tables

### erp_cust_az12

In [116]:
erp_cust_az12=pd.read_sql('''SELECT *
                          FROM raw.erp_cust_az12''',engine)
erp_cust_az12.head()

,cid,bdate,gen
0,NASAW00011000,1971-10-06,Male
1,NASAW00011001,1976-05-10,Male
2,NASAW00011002,1971-02-09,Male
3,NASAW00011003,1973-08-14,Female
4,NASAW00011004,1979-08-05,Female


In [117]:
erp_cust_az12['gen'].value_counts(dropna=False)

gen
Male      8608
Female    8391
None      1472
F            4
             3
M            3
             1
M            1
F            1
Name: count, dtype: int64

In [118]:
erp_cust_az12[erp_cust_az12['bdate']>date.today()]

,cid,bdate,gen
257,NASAW00011257,2050-07-06,Female
410,NASAW00011410,2042-02-22,Male
551,NASAW00011551,2050-05-21,Male
562,NASAW00011562,2038-10-17,Male
581,NASAW00011581,2045-03-03,Female
775,NASAW00011775,2050-11-22,Female
912,NASAW00011912,2066-06-16,Female
915,NASAW00011915,9999-09-13,Male
1123,NASAW00012123,2065-12-12,Male
1188,NASAW00012188,9999-11-20,Female


In [119]:
## clean id for future join
erp_cust_az12["cid"] = erp_cust_az12["cid"].apply(
    lambda x: x[3:] if pd.notna(x) and str(x).startswith("NAS") else x)
## normaliza gender
erp_cust_az12["gen"] = (erp_cust_az12["gen"].astype("string").str.strip().str.upper()
    .map({
        "F": "Female",
        "FEMALE": "Female",
        "M": "Male",
        "MALE": "Male"
    }).fillna("Unknown"))

## remove future birthdays
erp_cust_az12["bdate"] = erp_cust_az12["bdate"].mask(
    erp_cust_az12["bdate"] > date.today()
)

In [121]:
with engine.begin() as conn:
    print('Truncating Table: erp_cust_az12')
    conn.execute(text("TRUNCATE TABLE raw.erp_cust_az12"))

    print('Loading Table: erp_cust_az12')
    erp_cust_az12.to_sql(name='erp_cust_az12',schema='clean',con=conn,if_exists='append',index=False)

Truncating Table: erp_cust_az12
Loading Table: erp_cust_az12


### erp_loc_a101

In [ ]:
erp_loc_a101=pd.read_sql('''SELECT *
                          FROM raw.erp_loc_a101''',engine)
erp_loc_a101.head()

,cid,cntry
0,AW-00011000,Australia
1,AW-00011001,Australia
2,AW-00011002,Australia
3,AW-00011003,Australia
4,AW-00011004,Australia


In [127]:
erp_loc_a101['cntry'].value_counts(dropna=False)

cntry
Australia         3591
United States     3391
USA               2591
United Kingdom    1913
France            1810
Canada            1571
US                1500
Germany           1214
DE                 566
None               332
                     2
                     2
                     1
Name: count, dtype: int64

In [124]:
erp_loc_a101['cid']=erp_loc_a101['cid'].astype(str).str.replace('-','')
erp_loc_a101["cntry"] = erp_loc_a101.apply(lambda x:
        "Germany" if  str(x["cntry"]).strip().upper() == "DE" else 
        "United States" if str(x["cntry"]).strip().upper() in ("US", "USA") else 
        "Unknown" if pd.isna(x["cntry"]) or str(x["cntry"]).strip() == "" else 
        str(x["cntry"]).strip(),
    axis=1
)

In [128]:
with engine.begin() as conn:
    print('Truncating Table: erp_loc_a101')
    conn.execute(text("TRUNCATE TABLE raw.erp_loc_a101"))

    print('Loading Table: erp_loc_a101')
    erp_loc_a101.to_sql(name='erp_loc_a101',schema='clean',con=conn,if_exists='append',index=False)

Truncating Table: erp_loc_a101
Loading Table: erp_loc_a101


### erp_px_cat_g1v2

In [129]:
erp_px_cat_g1v2=pd.read_sql('''SELECT *
                          FROM raw.erp_px_cat_g1v2''',engine)
erp_px_cat_g1v2.head()

,id,cat,subcat,maintenance
0,AC_BR,Accessories,Bike Racks,Yes
1,AC_BS,Accessories,Bike Stands,No
2,AC_BC,Accessories,Bottles and Cages,No
3,AC_CL,Accessories,Cleaners,Yes
4,AC_FE,Accessories,Fenders,No


In [130]:
erp_px_cat_g1v2['cat'].value_counts(dropna=False)

cat
Components     14
Accessories    12
Clothing        8
Bikes           3
Name: count, dtype: int64

In [131]:
erp_px_cat_g1v2['subcat'].value_counts(dropna=False)

subcat
Bike Racks           1
Bike Stands          1
Bottles and Cages    1
Cleaners             1
Fenders              1
Helmets              1
Hydration Packs      1
Lights               1
Locks                1
Panniers             1
Pumps                1
Tires and Tubes      1
Mountain Bikes       1
Road Bikes           1
Touring Bikes        1
Bib-Shorts           1
Caps                 1
Gloves               1
Jerseys              1
Shorts               1
Socks                1
Tights               1
Vests                1
Bottom Brackets      1
Brakes               1
Chains               1
Cranksets            1
Derailleurs          1
Forks                1
Handlebars           1
Headsets             1
Mountain Frames      1
Pedals               1
Road Frames          1
Saddles              1
Touring Frames       1
Wheels               1
Name: count, dtype: int64

In [132]:
erp_px_cat_g1v2['maintenance'].value_counts(dropna=False)

maintenance
Yes    20
No     17
Name: count, dtype: int64

In [133]:
with engine.begin() as conn:
    print('Truncating Table: erp_px_cat_g1v2')
    conn.execute(text("TRUNCATE TABLE raw.erp_px_cat_g1v2"))

    print('Loading Table: erp_px_cat_g1v2')
    erp_px_cat_g1v2.to_sql(name='erp_px_cat_g1v2',schema='clean',con=conn,if_exists='append',index=False)

Truncating Table: erp_px_cat_g1v2
Loading Table: erp_px_cat_g1v2
